# Explainability, export, and ONNX parity

Privacy rule: no subject, body, preview, or attachment content is loaded.

In [ ]:
PROJECT_ROOT = '..'
RANDOM_SEED = 365

In [ ]:
from pathlib import Path
ROOT = Path(PROJECT_ROOT).resolve()
DATA_ROOT = ROOT / 'data'
ARTIFACT_ROOT = ROOT / 'artifacts/models/current'

In [ ]:
import json, pandas as pd
manifest = json.loads((ARTIFACT_ROOT/'manifest.json').read_text())
metrics = json.loads((ARTIFACT_ROOT/'metrics.json').read_text())
display(pd.DataFrame([manifest]))
print('ONNX max absolute difference:', metrics['onnx_max_abs_difference'])

In [ ]:
coefficients = pd.read_csv(ARTIFACT_ROOT/'coefficients.csv').sort_values('coefficient')
coefficients.plot.barh(x='feature', y='coefficient', figsize=(9,6), title='Calibrated model coefficients')

In [ ]:
import joblib, shap, numpy as np
from packages.ml.m365risk_ml.features import FEATURE_NAMES
pipeline=joblib.load(ARTIFACT_ROOT/'pipeline.joblib'); sample=pd.read_parquet(ARTIFACT_ROOT/'email_features.parquet',columns=FEATURE_NAMES).sample(80,random_state=RANDOM_SEED)
background=sample.iloc[:30]
def phishing_probability(values):
    return pipeline.predict_proba(pd.DataFrame(values,columns=FEATURE_NAMES))[:,1]
explainer=shap.Explainer(phishing_probability,background,algorithm='permutation',seed=RANDOM_SEED)
explanation=explainer(sample.iloc[30:60],max_evals=2*len(FEATURE_NAMES)+1); explanation.feature_names=FEATURE_NAMES
shap.plots.beeswarm(explanation,max_display=14)

In [ ]:
import hashlib
for name, expected in manifest['artifacts'].items():
    actual = hashlib.sha256((ARTIFACT_ROOT/name).read_bytes()).hexdigest()
    assert actual == expected, name
limit=1e-5 if manifest['selected_model']=='logistic' else 1e-4
assert metrics['onnx_max_abs_difference'] <= limit
print('All artifact checksums and parity gates verified')